In [ ]:
%%writefile ./endpoint.yaml
$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineEndpoint.schema.json
name: AllRecipeEndpoint
auth_mode: key

In [ ]:
!az ml online-endpoint create --file ./endpoint.yaml

In [ ]:
%%writefile ./deployment.yaml
$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json
name: blue
endpoint_name: AllRecipeEndpoint
model: azureml:phi3_finetuned_allrecipes@latest
code_configuration:
  code: .
  scoring_script: score_allrecipes.py
environment: azureml:phi35vision@latest
instance_type: Standard_NC12s_v3
instance_count: 1
request_settings:
  request_timeout_ms: 180000

In [3]:
!az ml online-deployment create --all-traffic --file ./deployment.yaml

......................................................................................................................................................{
  "app_insights_enabled": false,
  "code_configuration": {
    "code": "/subscriptions/781b03e7-6eb7-4506-bab8-cf3a0d89b1d4/resourceGroups/antonslutsky-rg/providers/Microsoft.MachineLearningServices/workspaces/gpu-workspace/codes/ed92f7bd-2a7a-49e3-b28d-d7d2b81cffdc/versions/1",
    "scoring_script": "score_allrecipes.py"
  },
  "egress_public_network_access": "enabled",
  "endpoint_name": "allrecipeendpoint",
  "environment": "azureml:/subscriptions/781b03e7-6eb7-4506-bab8-cf3a0d89b1d4/resourceGroups/antonslutsky-rg/providers/Microsoft.MachineLearningServices/workspaces/gpu-workspace/environments/phi35vision/versions/14",
  "environment_variables": {
    "AML_APP_ROOT": "/var/azureml-app/endpoint",
    "AZUREML_ENTRY_SCRIPT": "score_allrecipes.py",
    "AZUREML_MODEL_DIR": "/var/azureml-app/azureml-models/phi3_finetuned_allrecipes/1"
 

All traffic will be set to deployment blue once it has been provisioned.
If you interrupt this command or it times out while waiting for the provisioning, you can try to set all the traffic to this deployment later once its has been provisioned.
Check: endpoint AllRecipeEndpoint exists

Uploading endpoint (1.82 MBs): 100%|##########| 1817901/1817901 [00:01<00:00, 1339903.04it/s]




# Test the Deployment

In [8]:
import urllib.request
import json
import os
import ssl
from score_allrecipes import process_actions_string, image_to_data_url, load_image

# Replace this with the primary/secondary key, AMLToken, or Microsoft Entra ID token for the endpoint
api_key = 'dwgHkNprbQ8YJeq3YSN5mTrfoatcM2p4'
url = 'https://allrecipeendpoint.northeurope.inference.ml.azure.com/score'

def allowSelfSignedHttps(allowed):
    # bypass the server certificate verification on client side
    if allowed and not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
        ssl._create_default_https_context = ssl._create_unverified_context

allowSelfSignedHttps(True) # this line is needed if you use self-signed certificate in your scoring service.


images_root_dir = "C:/Users/antonslutsky/Dev/azureml-quickstart/slm-finetuning/phi3-vision-finetune/applications/AllrecipesAgent/output/session_2024-09-01_13-44-04/images"
images_ext = ".png"

def local_image_loader(local_image_name):
    return load_image(f"{images_root_dir}/{local_image_name}{images_ext}")

def local_image_action_updater(image, img_cnt, actions_image_url):
    image = local_image_loader(actions_image_url)
    return image_to_data_url(image, images_ext)


def test_slm(prompt = "You are a useful AI that searches AllRecipes.com website for various recipies.  The following document contains a set of keyboard and mouth actions together with the screenshots that preempted them to search for 'mashed potatoes' recipe on the website.  Suggest the nest set of keyboard and mouth actions to continue searching for the recipe. ['<sleep>11.645689', '<image>screenshot_2024-09-01_13-44-26.624344', 'mashed']"):

    action_string, images = process_actions_string(prompt, 
                                                   image_loader=local_image_loader,
                                                   action_updater=local_image_action_updater)
    
    action_string = action_string.replace("<|end|><|assistant|>", "").replace("data:image/png;base64", "data:image/jpeg;base64")

    with open("test_out.txt", "w") as test_out:
        test_out.write(action_string)

    data = {"input_data": {"input_string": [
            action_string
    ]}}
    
    #print("action_string:", action_string)

    body = str.encode(json.dumps(data))
    # print("body:", body)

    if not api_key:
        raise Exception("A key should be provided to invoke the endpoint")

    headers = {'Content-Type':'application/json', 'Authorization':('Bearer '+ api_key), 'azureml-model-deployment': 'blue' }

    if True:
        req = urllib.request.Request(url, body, headers)

        try:
            print("------------------------------------")
            response = urllib.request.urlopen(req)

            
            result = response.read()
            print(result)
            print("=====================================")
            #return result
        except urllib.error.HTTPError as error:
            print("The request failed with status code: " + str(error.code))

            # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
            print(error.info())
            print(error.read().decode("utf8", 'ignore'))


test_slm()

----------------------------------
INPUT_STRING: You are a useful AI that searches AllRecipes.com website for various recipies.  The following document contains a set of keyboard and mouth actions together with the screenshots that preempted them to search for 'mashed potatoes' recipe on the website.  Suggest the nest set of keyboard and mouth actions to continue searching for the recipe. ['<sleep>11.645689', '<image>screenshot_2024-09-01_13-44-26.624344', 'mashed']
----------------------------------
------------------------------------
b'[{"0": "[\\"<sleep>11.645689\\", \\"<image>]"}]'
